# Amazon Bedrock APIs — 

T

## Learning flow
1. Text generation
2. Synthetic sentiment test data
3. Amazon Nova request structure and `invoke_model`
4. Extractive vs. abstractive summarization
5. Batch summarization workflow and JSONL manifest
6. Question answering with documents
7. Amazon Bedrock `Converse` API
8. Referencing documents in Amazon S3

## AWS learner access

Local examples run without AWS credentials. For live model cells, use an AWS profile/SSO session or choose `AWS_AUTH_METHOD = "keys"` in the first code cell to enter instructor-provided temporary credentials through hidden prompts. The learner role needs `bedrock:InvokeModel` for the selected model. For the optional S3 document example, use your own bucket and allow `s3:GetObject` only on its lab prefix. See `setup/bedrock_course_roles.md` in the course repository.


In [ ]:
# Common imports
import json
import random
import os
import getpass
from collections import Counter

# Optional AWS SDK; local teaching cells can run without boto3.
try:
    import boto3
except ImportError:
    boto3 = None

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REGION = AWS_REGION
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.amazon.nova-lite-v1:0")
aws_session = None

if boto3 is not None:
    AWS_AUTH_METHOD = "profile"  # Change to "keys" only when a profile is unavailable.
    if AWS_AUTH_METHOD == "profile":
        AWS_PROFILE_NAME = os.getenv("AWS_PROFILE") or None
        aws_session = boto3.Session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
        boto3.setup_default_session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
        print("Using AWS profile/default credential chain.")
    elif AWS_AUTH_METHOD == "keys":
        print("Enter temporary credentials only. Inputs are hidden and are not saved in this notebook.")
        access_key_id = getpass.getpass("AWS access key ID: ")
        secret_access_key = getpass.getpass("AWS secret access key: ")
        session_token = getpass.getpass("AWS session token (leave blank if not applicable): ")
        credential_args = {
            "aws_access_key_id": access_key_id,
            "aws_secret_access_key": secret_access_key,
            "region_name": AWS_REGION,
        }
        if session_token:
            credential_args["aws_session_token"] = session_token
        aws_session = boto3.Session(**credential_args)
        boto3.setup_default_session(**credential_args)
        del access_key_id, secret_access_key, session_token, credential_args
        print("Using temporary credentials for this kernel session.")
    else:
        raise ValueError('AWS_AUTH_METHOD must be "profile" or "keys".')
else:
    print("boto3 is not installed; local examples remain available, but AWS examples are disabled.")

print("Region:", REGION)


##  Text generation

**Prompt input request → Foundation Model → Generated output**

Use this first example to establish that an application supplies instructions/content and the model generates text in response.


In [ ]:
# Local teaching example: construct a text-generation prompt
city = "Seattle"
prompt = f"""Write a short, positive travel description of {city}.
Mention food, culture, attractions, and lifestyle.
Keep the response under 120 words."""

print(prompt)


##  Generating sentiment test data

The deck uses a city-review application to demonstrate **synthetic test-data generation**.  
For each city, the application chooses a sentiment, builds a prompt and rating, invokes Amazon Nova, and can store the generated test record.

In [ ]:
CITIES = ["Seattle", "Austin", "Boston", "Chicago", "Miami"]

def generate_sentiment_distribution():
    # Teaching-friendly distribution; change weights to demonstrate test scenarios.
    return random.choices(
        ["positive", "neutral", "negative"],
        weights=[0.5, 0.3, 0.2],
        k=len(CITIES)
    )

def generate_review_prompt(city, sentiment):
    if sentiment == "positive":
        prompts = [
            f"Write an enthusiastic 4- or 5-star review of {city}. "
            "Focus on food, culture, attractions, and lifestyle. "
            "Make it personal and positive. Keep it to about 120 words.",
            f"Create a glowing review of {city}, highlighting neighborhoods, "
            "people, dining, and unique attractions. Keep it to about 120 words."
        ]
        rating = random.choice([4, 4, 5])
    elif sentiment == "neutral":
        prompts = [
            f"Write a balanced 3-star review of {city}. Mention both strengths "
            "and limitations. Keep it to about 120 words."
        ]
        rating = 3
    else:
        prompts = [
            f"Write a critical 1- or 2-star test review of {city}. Mention realistic "
            "travel frustrations without hateful or abusive content. Keep it to about 120 words."
        ]
        rating = random.choice([1, 2, 2])

    return random.choice(prompts), rating

sentiments = generate_sentiment_distribution()
for city, sentiment in zip(CITIES, sentiments):
    prompt, rating = generate_review_prompt(city, sentiment)
    print(f"\n{city}: sentiment={sentiment}, rating={rating}\nPrompt: {prompt}")


##  Invoke Amazon Nova with `invoke_model`

Build a Nova request using:
- a `messages` list
- `schemaVersion: messages-v1`
- inference parameters such as `maxTokens` and `temperature`
- `bedrock-runtime.invoke_model(...)`
- extraction of generated text from the response body


In [ ]:
def build_nova_body(prompt, max_tokens=1000, temperature=0.7):
    message_list = [
        {"role": "user", "content": [{"text": prompt}]}
    ]
    inf_params = {
        "maxTokens": max_tokens,
        "temperature": temperature
    }
    return {
        "schemaVersion": "messages-v1",
        "messages": message_list,
        "inferenceConfig": inf_params
    }

sample_prompt, _ = generate_review_prompt("Seattle", "positive")
body = build_nova_body(sample_prompt)

print(json.dumps(body, indent=2))


In [ ]:
# LIVE AWS CELL — requires AWS credentials and Bedrock model access.
# Uncomment the final lines when teaching in a configured AWS environment.

def call_nova(prompt, bedrock_client, model_id=MODEL_ID):
    body = build_nova_body(prompt)

    response = bedrock_client.invoke_model(
        modelId=model_id,
        body=json.dumps(body)
    )

    response_body = json.loads(response["body"].read())
    return response_body["output"]["message"]["content"][0]["text"]

# bedrock_client = aws_session.client("bedrock-runtime", region_name=REGION)
# print(call_nova(sample_prompt, bedrock_client))


##  Text summarization

The deck distinguishes two techniques:

- **Extractive summarization:** selects important words, phrases, or sentences from the source.
- **Abstractive summarization:** generates a new concise explanation of the source meaning.

The examples below make the distinction visible without requiring an API call.


In [ ]:
source_text = """Amazon Bedrock provides access to foundation models through managed APIs.
Developers can use these models for generation, summarization, question answering,
and conversational applications. Applications should choose an inference approach
based on latency, cost, and workload requirements. Batch inference is useful for
large asynchronous workloads, while on-demand inference supports interactive use cases."""

# Very simple extractive teaching demo:
sentences = [s.strip() for s in source_text.replace("\n", " ").split(".") if s.strip()]
extractive_summary = ". ".join(sentences[:2]) + "."

print("SOURCE:\n", source_text)
print("\nEXTRACTIVE SUMMARY:\n", extractive_summary)


In [ ]:
# Abstractive summarization prompt for Amazon Bedrock
abstractive_prompt = f"""Summarize the following text in 2 concise sentences.
Preserve the main meaning, but use your own wording.

TEXT:
{source_text}
"""

print(abstractive_prompt)

# Optional live call:
# print(call_nova(abstractive_prompt, bedrock_client))


## Slide 17 — Summarizing files

A file can be supplied as context when its content fits within the model's context window.  
For teaching, first demonstrate the idea with a small text file, then connect it to the Bedrock document examples in slides 20–23.


In [ ]:
# Create a small local file to represent a document being summarized.
sample_file = "sample_business_note.txt"
with open(sample_file, "w", encoding="utf-8") as f:
    f.write(
        "Customer feedback increased this quarter. "
        "Most customers praised faster support response times. "
        "The main complaint was difficulty finding billing information. "
        "The team plans to redesign the billing help pages next month."
    )

with open(sample_file, "r", encoding="utf-8") as f:
    file_text = f.read()

summary_prompt = f"""Summarize this business document in 3 bullet points:

{file_text}
"""

print(summary_prompt)


##  Batch summarization workflow

The deck's example uses an event-driven **AWS Step Functions** workflow with states such as:

`CreateBatchManifest → StartBatchInference → WaitState → CheckBatchStatus → DecisionPoint → ProcessBatchOutput`

A batch manifest is JSON Lines (**JSONL**): one model request per line. The slide example includes a system instruction, user content, and inference settings including `maxTokens`, `topP`, `topK`, and `temperature`.


In [ ]:
reviews_by_city = {
    "albuquerque": [
        "Great food and friendly people.",
        "The old town area was enjoyable, but traffic was difficult.",
        "Museums were interesting and the weather was pleasant."
    ],
    "seattle": [
        "Loved the waterfront and coffee shops.",
        "Public transport was convenient.",
        "Rainy weather affected some outdoor plans."
    ]
}

def create_batch_record(city, reviews):
    review_text = "\n".join(f"- {r}" for r in reviews)
    return {
        "recordId": city,
        "modelInput": {
            "schemaVersion": "messages-v1",
            "messages": [{
                "role": "user",
                "content": [{
                    "text": (
                        f"Use the following reviews for {city.title()}:\n{review_text}\n\n"
                        "Please provide a well-structured summary that includes:\n"
                        "1. Overall impression and sentiment\n"
                        "2. Common positive themes\n"
                        "3. Common concerns"
                    )
                }]
            }],
            "system": [{
                "text": "You are a travel expert AI assistant. Create comprehensive, engaging city summaries based on user reviews."
            }],
            "inferenceConfig": {
                "maxTokens": 500,
                "topP": 0.9,
                "topK": 20,
                "temperature": 0.7
            }
        }
    }

manifest_path = "city_summarization_manifest.jsonl"
with open(manifest_path, "w", encoding="utf-8") as f:
    for city, reviews in reviews_by_city.items():
        f.write(json.dumps(create_batch_record(city, reviews)) + "\n")

print(open(manifest_path, encoding="utf-8").read())


## Slide 20 — Question answering with relevant context

The pattern is:

**User question + relevant context → prompt request → FM → generated answer**

Context can be file contents or references to files in Amazon S3. This turns general generation into context-grounded question answering / information extraction.


In [ ]:
context = """Project Aurora began in January.
The first prototype was completed in March.
Production launch is planned for October.
The project owner is the Applied AI team."""

question = "When is Project Aurora planned to launch?"

qa_prompt = f"""Answer the question using only the supplied context.
If the answer is not in the context, say that it is not available.

CONTEXT:
{context}

QUESTION:
{question}
"""

print(qa_prompt)

# Optional:
# print(call_nova(qa_prompt, bedrock_client))


## Slides 21–22 — Document Q&A with the Converse API

The deck reads PDF bytes, adds a `document` block plus a text question to the user message, and sends the `messages` object through `bedrock-runtime.converse()`.

The inference example uses `maxTokens`, `topP`, and `temperature`.


In [ ]:
# LIVE AWS DOCUMENT-Q&A TEMPLATE
# Set PDF_PATH to a PDF available in your notebook environment.

PDF_PATH = "my_document.pdf"

def build_document_messages(pdf_path, question):
    with open(pdf_path, "rb") as file:
        doc_bytes = file.read()

    return [{
        "role": "user",
        "content": [
            {
                "document": {
                    "format": "pdf",
                    "name": "DocumentPDFmessages",
                    "source": {"bytes": doc_bytes}
                }
            },
            {"text": question}
        ]
    }]

# Example:
# messages = build_document_messages(
#     PDF_PATH,
#     "How many qubits of growth is projected by 2026 by the industry, and how does the actual trajectory differ?"
# )


In [ ]:
# Converse API call corresponding to slide 22
def converse_with_document(messages):
    if boto3 is None:
        raise RuntimeError("Install boto3 first.")

    client = aws_session.client("bedrock-runtime", region_name=REGION)
    inf_params = {
        "maxTokens": 300,
        "topP": 0.1,
        "temperature": 0.3
    }

    model_response = client.converse(
        modelId=MODEL_ID,
        messages=messages,
        inferenceConfig=inf_params
    )

    print("\n[Full Response]")
    print(json.dumps(model_response, indent=2, default=str))
    print("\n[Response Content Text]")
    print(model_response["output"]["message"]["content"][0]["text"])
    return model_response

# model_response = converse_with_document(messages)


## Slide 23 — Question answering with an Amazon S3 document reference

Instead of reading the document into application memory, the message can reference an object in Amazon S3.  
The slide notes this as an alternative for large or multiple documents, while the Bedrock client invocation and response-processing pattern remains similar.


In [ ]:
s3_messages = [{
    "role": "user",
    "content": [
        {
            "document": {
                "format": "pdf",
                "name": "sample_doc",
                "source": {
                    "s3Location": {
                        "uri": "s3://demo-bucket/document1.pdf",
                        "bucketOwner": "<AWS_ACCOUNT_ID>"
                    }
                }
            }
        },
        {"text": "Describe the following document"}
    ]
}]

print(json.dumps(s3_messages, indent=2))

# In a configured environment:
# converse_with_document(s3_messages)


## Instructor recap / discussion

Ask learners to explain which pattern they would choose:

1. Generate synthetic city reviews for testing → **Text generation**
2. Summarize thousands of city reviews every 12 hours → **Batch summarization**
3. Answer a question using a PDF → **Document question answering**
4. Avoid loading a large document directly into application memory → **S3 document reference**
5. Need a unified message-oriented API for document + question input → **Converse API**

### Mini challenge
Modify the notebook to:
- generate three synthetic reviews,
- build a JSONL batch manifest from them,
- create a summarization prompt,
- then create a question that can be answered from the generated summary.
